# Product Review Sentiment Analysis Using LSTM with GloVe Embeddings

This interactive notebook demonstrates the complete Deep Learning pipeline for analyzing customer sentiment from Flipkart product reviews.

### Pipeline Overview:
1. **Dataset Ingestion & Inspection**: Flipkart product reviews dataset (~190k records).
2. **Data Cleaning & Normalization**: HTML stripping, contraction expansion, duplicate removal.
3. **Pre-trained GloVe Embeddings**: Stanford 100-dimensional GloVe vectors alignment.
4. **LSTM Architecture**: Tokenizer, GloVe Embedding Layer, LSTM, Dropout, Dense, Softmax.
5. **Evaluation**: Holdout metrics, Confusion Matrix, Classification Report.
6. **Inference**: Live review predictions.

In [ ]:
import os
import sys
# Ensure project root is in sys.path
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_raw_dataset, inspect_dataset, detect_columns, prepare_labeled_data, split_dataset
from src.preprocessing import clean_text, TextPreprocessor
from src.glove_embeddings import load_glove_vectors, build_embedding_matrix
from src.model import build_sentiment_lstm_model

## 1. Load and Inspect Flipkart Dataset

In [ ]:
csv_path = '../dataset/flipkart_product.csv'
df_raw, encoding = load_raw_dataset(csv_path)
print(f"Loaded with encoding '{encoding}': shape = {df_raw.shape}")
df_raw.head(3)

## 2. Text Cleaning & Label Mapping

Ratings are mapped to 3 classes:
- 1 and 2 Stars &rarr; Negative
- 3 Stars &rarr; Neutral
- 4 and 5 Stars &rarr; Positive

In [ ]:
text_cols, rating_col = detect_columns(df_raw)
df_clean = prepare_labeled_data(df_raw, text_cols=text_cols, rating_col=rating_col, sample_size=10000)
print("Sample cleaned dataset:")
df_clean.head(5)

## 3. Stanford GloVe Word Embeddings Alignment

In [ ]:
glove_path = '../embeddings/glove.6B.100d.txt'
if os.path.exists(glove_path):
    glove_index, dim = load_glove_vectors(glove_path, expected_dim=100)
    print(f"Loaded {len(glove_index):,} GloVe vectors of dimension {dim}")
else:
    print("GloVe file not found. Run python -m src.download_glove 100 to download.")

## 4. Live Prediction with Pre-trained Artifacts

In [ ]:
from src.predict import predict_sentiment

sample_review = "The build quality is outstanding and battery backup lasts all day."
try:
    res = predict_sentiment(sample_review, artifacts_dir='../artifacts')
    print("Sentiment:", res['predicted_sentiment'])
    print("Confidence:", f"{res['confidence']*100:.1f}%")
    print("Probabilities:", res['probabilities'])
except Exception as e:
    print("Train model first to run live inference:", e)